# Modelo LLM local con selección de Tools

In [1]:
from openai import OpenAI

In [2]:
cliente = OpenAI(
    base_url="http://localhost:8080/v1",
    api_key="local"
)

In [3]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "calcular_interes",
            "description": "Calcula el interés generado por un capital",
            "parameters": {
                "type": "object",
                "properties": {
                    "capital": {
                        "type": "number"
                    },
                    "tasa": {
                        "type": "number"
                    }
                },
                "required": [
                    "capital",
                    "tasa"
                ]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calcular_inflacion",
            "description": "Calcula la inflación generada por un producto",
            "parameters": {
                "type": "object",
                "properties": {
                    "precio_anterior": {
                        "type": "number"
                    },
                    "precio_nuevo": {
                        "type": "number"
                    }
                },
                "required": [
                    "precio_anterior",
                    "precio_nuevo"
                ]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "fallar_respuesta",
            "description": "Falla la respuesta si no logras capturar los cálculos",
            "parameters": {
                "type": "object",
                "properties": {
                    "mensaje_original": {
                        "type": "string"
                    },
                    "razon_fallo": {
                        "type": "string"
                    }
                }
            }
        }
    },
]

In [5]:
respuesta = cliente.chat.completions.create(
    model="local",
    messages=[
        {
            "role": "user",
            "content": "¿Cuánto genera $15000 a una tasa de 8%?"
        }
    ],
    tools=tools,
    tool_choice="auto"
)


message = respuesta.choices[0].message

print(message.tool_calls[0].function.name)
print(message.tool_calls[0].function.arguments)

calcular_interes
{"capital": 15000, "tasa": 0.08}


In [6]:
respuesta = cliente.chat.completions.create(
    model="local",
    messages=[
        {
            "role": "user",
            "content": "¿Cuál fue la inflación de un producto que costaba $3,500.53 y ahora cuesta $6,275.79?"
        }
    ],
    tools=tools,
    tool_choice="auto"
)


message = respuesta.choices[0].message

print(message.tool_calls[0].function.name)
print(message.tool_calls[0].function.arguments)

calcular_inflacion
{"precio_anterior": 3500.53, "precio_nuevo": 6275.79}


In [16]:
respuesta = cliente.chat.completions.create(
    model="local",
    messages=[
        {
            "role": "user",
            "content": "¿Qué es la inflación?"
        }
    ],
    tools=tools,
    tool_choice="auto"
)

try:
    message = respuesta.choices[0].message

    print(message.tool_calls[0].function.name)
    print(message.tool_calls[0].function.arguments)
except:
    print("No se encontró el tool adecuado")
    print("-" * 80)
    print(respuesta.choices[0].message.content)

No se encontró el tool adecuado
--------------------------------------------------------------------------------
La inflación es un aumento generalizado en los precios de los bienes y servicios en una economía durante una período de tiempo. Esto significa que el poder adquisitivo del dinero se reduce con el tiempo, ya que cada unidad de dinero (por ejemplo, cada dólar o euro) puede comprar menos bienes y servicios que antes. La inflación se mide normalmente usando la tasa de inflación, que puede ser medida en términos anuales, mensuales o porcentualmente.
